In [26]:
import os
import time
import requests
from bs4 import BeautifulSoup
from markdownify import markdownify as md
from urllib.parse import urljoin

In [27]:
# --- CẤU HÌNH ---
BASE_URL = "http://fit.hcmute.edu.vn"
OUTPUT_FILE = "DATA_WEB_FIT.md"
visited_urls = set()

if os.path.exists(OUTPUT_FILE):
    os.remove(OUTPUT_FILE)
    print(f"Đã dọn dẹp file cũ. Sẵn sàng ghi dữ liệu mới vào: {OUTPUT_FILE}")
else:
    print(f"Đã sẵn sàng tạo file mới: {OUTPUT_FILE}")

Đã dọn dẹp file cũ. Sẵn sàng ghi dữ liệu mới vào: DATA_WEB_FIT.md


In [28]:
def crawl_website_single_file(url, max_depth=5, current_depth=0):
    url_clean = url.split("#")[0] 
        
    # Điều kiện dừng: Đã thăm rồi, hoặc khác tên miền, hoặc đi quá sâu
    if url_clean in visited_urls or not url.startswith(BASE_URL) or current_depth > max_depth:
        return
    
    print(f"[{current_depth}/{max_depth}] Đang thu thập: {url_clean}")
    visited_urls.add(url_clean)

    try:
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
        response = requests.get(url_clean, headers=headers, timeout=10)
        response.encoding = 'utf-8'
        if response.status_code == 200:
            soup = BeautifulSoup(response.content, 'html.parser')
            main_content = soup.find('div', id='pnCenter')
            
            # --- LẤY NỘI DUNG VÀ DỌN RÁC ---
            if main_content:
                text = md(str(main_content), heading_style="ATX").strip()
                
                # Cắt rác
                for keyword in ["Các tin khác", "Họ và tên:", "Góp ý"]:
                    if keyword in text:
                        text = text.split(keyword)[0]
                text = text.strip()
                
                if text: 
                    with open(OUTPUT_FILE, "a", encoding="utf-8") as f:
                        f.write(f"\n\n{'='*60}\n")
                        f.write(f"URL: {url_clean}\n")
                        f.write(f"{'='*60}\n\n")
                        f.write(text)
            
            # --- QUÉT MỌI NGÓC NGÁCH TRÊN WEB (Kể cả dropdown menu) ---
            if current_depth < max_depth:
                for link in soup.find_all('a', href=True):
                    next_href = link.get('href').strip()
                    invalid_exts = ['.pdf', '.doc', '.docx', '.rar', '.zip', '.jpg', '.png']
                    
                    # Bỏ qua link ảo và link gọi hàm JS
                    if not any(next_href.lower().endswith(ext) for ext in invalid_exts) \
                       and not next_href.startswith(('javascript:', 'mailto:', 'tel:')):
                        
                        next_url = urljoin(BASE_URL, next_href)
                        crawl_website_single_file(next_url, max_depth, current_depth + 1)
                    
            time.sleep(1) 
            
    except Exception as e:
        print(f"[-] Lỗi tại {url_clean}: {e}")

In [29]:
# Làm sạch bộ nhớ
visited_urls.clear()

start_time = time.time()

# Để max_depth = 5 hoặc 10 để bot đào sâu vào từng bài viết trong mục
crawl_website_single_file(BASE_URL, max_depth=5) 

end_time = time.time()
print("-" * 40)
print(f"Done")
print(f"Total Time: {round(end_time - start_time, 2)} giây.")

[0/5] Đang thu thập: http://fit.hcmute.edu.vn
[1/5] Đang thu thập: http://fit.hcmute.edu.vn/Portlets/MenuRad/
[1/5] Đang thu thập: http://fit.hcmute.edu.vn/Default.aspx?ArticleId=bafd29bd-264e-4a5d-a92f-afe11c777e95
[2/5] Đang thu thập: http://fit.hcmute.edu.vn/Default.aspx?ArticleId=2c95115f-657e-46ea-9e67-5dfe411e1f9d
[3/5] Đang thu thập: http://fit.hcmute.edu.vn/Default.aspx?ArticleId=e1f1638d-58b1-4baa-a3d2-089d3b929788
[4/5] Đang thu thập: http://fit.hcmute.edu.vn/Default.aspx?ArticleId=4e268133-b495-4b0e-9b5c-1d0aa72d5605
[5/5] Đang thu thập: http://fit.hcmute.edu.vn/Default.aspx?ArticleId=f3944f07-6d3d-42a2-bd21-0b97f0f3d37c
[5/5] Đang thu thập: http://fit.hcmute.edu.vn/Default.aspx?ArticleId=61077cd8-4278-4e09-ac91-a03925d4fa9f
[5/5] Đang thu thập: http://fit.hcmute.edu.vn/Default.aspx?ArticleId=bc6af40c-ef24-4f66-aa30-a6b1cac206c9
[5/5] Đang thu thập: http://fit.hcmute.edu.vn/Default.aspx?ArticleId=9634f3b8-d349-4168-a00e-714611df7e7d
[5/5] Đang thu thập: http://fit.hcmute.edu

Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


[5/5] Đang thu thập: http://fit.hcmute.edu.vn/?ArticleId=719ac8eb-2e74-4d25-8934-de3fdb929b90
[5/5] Đang thu thập: http://fit.hcmute.edu.vn/?ArticleId=545ea74e-3b9e-4297-8c7c-9312d9d9f40a
[5/5] Đang thu thập: http://fit.hcmute.edu.vn/?ArticleId=11f2efc3-0aed-4ece-b0f8-64683298537b
[5/5] Đang thu thập: http://fit.hcmute.edu.vn/?ArticleId=a05d3615-c033-4c78-98de-aa51c01377bc
[5/5] Đang thu thập: http://fit.hcmute.edu.vn/?ArticleId=e18f1c77-218e-46eb-b1ed-0eb2fbfb4a27
[5/5] Đang thu thập: http://fit.hcmute.edu.vn/?ArticleId=e550248f-11b4-497b-b122-701e6f1fd01d
[5/5] Đang thu thập: http://fit.hcmute.edu.vn/?ArticleId=8fd6c778-7396-4ecd-9582-0b9a6a219ee5
[5/5] Đang thu thập: http://fit.hcmute.edu.vn/?ArticleId=5ccca14c-867a-4f85-8d1a-642764f14088
[5/5] Đang thu thập: http://fit.hcmute.edu.vn/?ArticleId=bcbb1f5b-e329-40fa-9e61-2054149541b4
[5/5] Đang thu thập: http://fit.hcmute.edu.vn/?ArticleId=d2c4f27a-e8bc-4366-a431-8b19edc7d030
[1/5] Đang thu thập: http://fit.hcmute.edu.vn/?TopicId=a8b6c